# verl 框架架构速览

verl 是一个基于 Ray 的 **Hybrid Engine** RL 训练框架。在深入训练之前，先理解几个核心概念。

---

## 1. 整体架构

<img src="./images/verl_architecture.png" alt="verl 框架架构" width="80%">

图中把 verl 的 RL 训练流程按职责分成五层：

**调度层**：`Prompt 数据` 提供训练输入；`TaskRunner` 是训练主循环，负责发起 rollout、触发奖励计算、调度训练更新，并根据指标进入下一步。

**Rollout / 推理层**：`AgentLoopManager` 分发 rollout 任务；多个 `AgentLoop Worker` 并行执行多轮交互。以 Wordle 为例，Worker 会发送 prompt、接收 `<guess>[word]</guess>`、把 G/Y/X 反馈追加回对话，并重复直到猜中或达到最大轮次。`vLLM` 负责实际生成模型回复，和 AgentLoop 之间是请求 / 响应关系。

**奖励与优势层**：`Reward` 根据规则给每条 rollout 打分；`Advantage` 在同一个 prompt 的多条 rollout 内做相对比较，为 GRPO 更新提供优势信号。

**训练更新层**：`Ray Train / FSDP` 使用 rollout、reward 和 advantage 更新 Actor 参数；`Metrics / Checkpoint` 记录训练指标和 checkpoint，并把状态回传给调度层。

**资源复用层**：`Hybrid Engine` 负责在训练阶段和推理阶段之间切换资源。训练时 FSDP 占用设备，推理时 vLLM 占用同一组设备，因此 2 卡环境也能完成 RL 训练。

图中的线型含义如下：黑色虚线表示调度或状态回传，蓝色实线表示数据流，绿色点线表示资源复用关系。

---

## 2. Hybrid Engine：训推时分复用

上图底部的“资源复用层”给出了 Hybrid Engine 的核心思想：训练和推理不是同时占用设备，而是交替使用同一组卡。下面用 step 时间线补充它的执行顺序：


```text
卡0+1:  [Actor 更新] → vLLM wake / 同步权重 → [rollout 推理] → vLLM sleep → [下一次 Actor 更新]
           ↑ FSDP                     ↑ vLLM TP=2             ↑ 释放 rollout KV cache/权重
```

`wake` / `sleep` 控制的是 **vLLM rollout 引擎**：rollout 前唤醒并同步最新 Actor 权重，rollout 后释放 KV cache，并按 sleep level 释放 rollout 权重。`actor.fsdp_config.param_offload=True` 则是另一项独立配置，用于把训练侧 FSDP 参数卸载到 CPU；它不是 vLLM `sleep` 的含义。

因此，Hybrid Engine 解决的是**训练与推理资源切换和权重同步**问题，使同一组 NPU 能在不同阶段分别执行 Actor 更新和 rollout。

---

## 3. 关键配置参数

| 参数 | 含义 | 默认值 |
|------|------|--------|
| `rollout.n` | 每个 prompt 的 rollout 数（GRPO 组大小） | 8 |
| `train_batch_size` | 每步处理的 prompt 数 | 64 |
| `rollout.tensor_model_parallel_size` | vLLM 张量并行度 | 2 |
| `multi_turn.max_user_turns` | Wordle 最大猜词轮次 | 6 |
| `actor.optim.lr` | Actor 学习率 | 1e-6 |
| `actor.fsdp_config.param_offload` | 训练侧 FSDP 参数 CPU 卸载 | True |

---

## 课后练习


1. （判断题）verl 的 Hybrid Engine 允许训练和推理在同一组 NPU 上交替运行，因此 2 卡即可完成 RL 训练。

2. （判断题）AgentLoop 是多轮交互的逻辑单元，WordleAgentLoop 负责发送游戏反馈（G/Y/X）给模型。

3. （判断题）`param_offload=True` 表示训练参数始终保留在 GPU 显存中，不卸载到 CPU。

4. （单选题）在 verl 架构中，哪个组件负责生成模型的回复（rollout）？
    A. TaskRunner
    B. vLLM Server
    C. Ray Train (FSDP)
    D. AgentLoopManager

5. （单选题）`rollout.n=8` 表示什么含义？
    A. 使用 8 张卡进行推理
    B. 每个 prompt 生成 8 条 rollout
    C. 训练 batch size 为 8
    D. 最大交互轮次为 8

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/01_environment_setup/answer/01.03_answer.txt